<a href="https://colab.research.google.com/github/Shravani0526/Student_Performance_Prediction/blob/main/DeepFakeDetector.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install tensorflow opencv-python matplotlib pillow numpy gradio

In [ ]:
import tensorflow as tf
import numpy as np
import cv2
import gradio as gr
import matplotlib.pyplot as plt
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.preprocessing.image import img_to_array


In [ ]:
model = MobileNetV2(weights="imagenet")
last_conv_layer = model.get_layer("Conv_1")

14536120/14536120 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
def get_last_conv_layer(model):
    for layer in reversed(model.layers):
        try:
            # Conv layers output 4D tensors: (batch, height, width, channels)
            if len(layer.output.shape) == 4:
                return layer.name
        except:
            continue
    raise ValueError("No convolutional layer found")


In [ ]:
last_conv_layer_name = get_last_conv_layer(model)
print("Using last conv layer:", last_conv_layer_name)

Using last conv layer: out_relu


In [ ]:
def analyze_image(image):
    try:
        if image is None:
            return None, "No image uploaded."

        image = image.astype(np.uint8)

        resized = cv2.resize(image, (224, 224))
        img_array = img_to_array(resized)
        img_array = np.expand_dims(img_array, axis=0)
        img_array = preprocess_input(img_array)

        preds = model.predict(img_array)
        confidence = float(np.max(preds))

        label = "Likely Authentic" if confidence < 0.55 else "Potentially Manipulated"
        trust_score = round(confidence, 3)

        grad_model = tf.keras.models.Model(
            [model.inputs],
            [
                model.get_layer(last_conv_layer_name).output,
                model.output
            ]
        )

        with tf.GradientTape() as tape:
            conv_outputs, predictions = grad_model(img_array)
            loss = tf.reduce_max(predictions)

        grads = tape.gradient(loss, conv_outputs)
        pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

        conv_outputs = conv_outputs[0]
        heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
        heatmap = tf.squeeze(heatmap)

        heatmap = np.maximum(heatmap, 0)
        heatmap /= np.max(heatmap) + 1e-8

        heatmap = cv2.resize(heatmap, (224, 224))
        heatmap = np.uint8(255 * heatmap)
        heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)

        overlay = cv2.addWeighted(resized, 0.6, heatmap, 0.4, 0)

        return overlay, f"Trust Score: {trust_score} | Result: {label}"

    except Exception as e:
        return None, f"Error occurred: {str(e)}"


In [ ]:
interface = gr.Interface(
    fn=analyze_image,
    inputs=gr.Image(type="numpy", label="Upload Image"),
    outputs=[
        gr.Image(label="Suspicious Regions (Grad-CAM)"),
        gr.Textbox(label="Analysis Result")
    ],
    title="Deep-Fake Image Detector",
    description="AI-powered image authenticity analysis using pretrained CNN and explainable AI."
)

interface.launch(debug=True)


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://548f6ce38d0687cb24.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://548f6ce38d0687cb24.gradio.live
